1) Надо взять рандомного пользователя из тестовой выборки.
2) Найти похожих пользователей, которые смотрели фильмы, которые смотрел мой юзер.
3) Порекомендовать юзеру фильмы, которые были популярны у других юзеров.

Загрузка датасетов

In [80]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from typing import List
from tqdm import tqdm
import random
from scipy.sparse import csr_matrix

In [81]:
df_users = pd.read_csv('data_original/data_original/users.csv')
df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
df_items = pd.read_csv('data_original/data_original/items.csv')

In [82]:
df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'], errors='coerce')

In [83]:
df_interactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5476251 entries, 0 to 5476250
Data columns (total 5 columns):
 #   Column         Dtype         
---  ------         -----         
 0   user_id        int64         
 1   item_id        int64         
 2   last_watch_dt  datetime64[ns]
 3   total_dur      int64         
 4   watched_pct    float64       
dtypes: datetime64[ns](1), float64(1), int64(3)
memory usage: 208.9 MB


In [84]:
df_interactions['completed'] = (df_interactions['watched_pct'] >= 50).astype(int)

Время отбора, тестовый и тренировочный датасет.

In [85]:
test_size_days=10

In [86]:
from datetime import datetime, timedelta

# Тестовый промежуток времени равен 10 дней
max_date = df_interactions['last_watch_dt'].max()
test_start = max_date - timedelta(days=test_size_days)

In [87]:
# Разделение на тренировочный и тестовый
df_interactions_train = df_interactions[df_interactions['last_watch_dt'] < test_start]
df_interactions_test = df_interactions[df_interactions['last_watch_dt'] >= test_start]

In [88]:
df_interactions_train.info

<bound method DataFrame.info of          user_id  item_id last_watch_dt  total_dur  watched_pct  completed
0         176549     9506    2021-05-11       4250         72.0          1
1         699317     1659    2021-05-29       8317        100.0          1
2         656683     7107    2021-05-09         10          0.0          0
3         864613     7638    2021-07-05      14483        100.0          1
4         964868     9506    2021-04-30       6725        100.0          1
...          ...      ...           ...        ...          ...        ...
5476243   497899     9629    2021-05-29         45          1.0          0
5476244   438585     7829    2021-08-02       6804        100.0          1
5476245   786732     4880    2021-05-12        753          0.0          0
5476247   546862     9673    2021-04-13       2308         49.0          0
5476249   384202    16197    2021-04-19       6203        100.0          1

[4811285 rows x 6 columns]>

In [89]:
random_user = df_interactions_test.sample(1)  
random_user

,user_id,item_id,last_watch_dt,total_dur,watched_pct,completed
3490187,508249,10498,2021-08-13,652,12.0,0


In [90]:
random_user_id = random_user["user_id"].values[0]

In [91]:
watched_items = df_interactions_train[df_interactions_train["user_id"] == random_user_id]["item_id"].unique()
watched_items


array([ 5434,  7556, 10440, 12173,  7571,  4685,   101,  7834,  3804,
        8821, 12609, 10152,  9728,  5754, 11057,  1287, 15221,  3076,
        7829,  9900,  8437, 15580, 13855,  5693])

In [92]:
df_completed = df_interactions[df_interactions['completed'] == 1]


In [93]:
unique_users = df_completed['user_id'].unique()
unique_items = df_completed['item_id'].unique()

In [94]:
user_to_idx = {user: idx for idx, user in enumerate(unique_users)}
item_to_idx = {item: idx for idx, item in enumerate(unique_items)}

In [95]:
n_users = len(unique_users)
n_items = len(unique_items)

In [96]:
# user_item_matrix = df_interactions_train.pivot_table(
#     index="user_id",
#     columns="item_id",
#     values="completed",   # будем использовать бинарный признак (смотрел/не смотрел)
#     fill_value=0
# )

# sparse_matrix = csr_matrix(user_item_matrix.values)








In [97]:
# # --- 2. Находим индекс нашего пользователя ---
# target_index = user_item_matrix.index.get_loc(random_user_id)

In [98]:
# # --- 3. Считаем косинусное сходство ---
# similarities = cosine_similarity(sparse_matrix[target_index], sparse_matrix).flatten()

In [99]:
# # --- 4. Собираем в таблицу ---
# similar_users = pd.DataFrame({
#     "user_id": user_item_matrix.index,
#     "similarity": similarities
# })


In [100]:
# # Убираем самого пользователя
# similar_users = similar_users[similar_users["user_id"] != random_user_id]

# # Сортируем по схожести
# similar_users = similar_users.sort_values(by="similarity", ascending=False)

# similar_users.head(20)

In [101]:
from scipy.sparse import coo_matrix, csr_matrix

def build_user_item_sparse(df, user_col='user_id', item_col='item_id', value_col=None, dtype=np.uint8):
    # value_col = None -> бинарная матрица (1 = смотрел)
    if value_col is None:
        data = np.ones(len(df), dtype=dtype)
    else:
        # если есть непрерывная величина (rating/ watched_pct) — используем её
        data = df[value_col].astype(np.float32).values
        dtype = np.float32

    users = df[user_col].values
    items = df[item_col].values

    # уникальные пользователи/элементы + индексы
    uniq_users, inverse_users = np.unique(users, return_inverse=True)
    uniq_items, inverse_items = np.unique(items, return_inverse=True)

    # coo -> csr (это экономит память по сравнению с pivot + dense array)
    X = coo_matrix((data, (inverse_users, inverse_items)),
                   shape=(len(uniq_users), len(uniq_items)),
                   dtype=dtype).tocsr()

    return X, uniq_users, uniq_items



In [102]:
X, uniq_users, uniq_items = build_user_item_sparse(df_interactions_train)
user_to_idx = {u:i for i,u in enumerate(uniq_users)}
item_to_idx = {it:i for i,it in enumerate(uniq_items)}

In [103]:
def get_similar_users_by_overlap(X, uniq_users, item_to_idx, target_user_id, watched_items=None, top_n=50):
    """
    X: csr_matrix (n_users x n_items) — бинарная или со значениями
    uniq_users: ndarray user_ids (len = n_users)
    item_to_idx: dict item_id -> column index
    target_user_id: id юзера (может быть и из теста)
    watched_items: iterable item_id, просмотренные целевым юзером (если None -> попробуем взять из X, если user присутствует в train)
    top_n: вернуть топ-N похожих пользователей
    """
    n_users = X.shape[0]

    # Получаем список индексов item'ов, которые смотрел target
    if watched_items is None:
        if target_user_id in user_to_idx:
            target_idx = user_to_idx[target_user_id]
            items_idx = X[target_idx].indices.tolist()  # быстро извлекает ненулевые колонки
        else:
            items_idx = []
    else:
        # фильтруем те фильмы, которых нет в словаре item_to_idx
        items_idx = [item_to_idx[i] for i in watched_items if i in item_to_idx]

    if len(items_idx) == 0:
        # у целевого пользователя нет просмотров в train (нет общей базы) -> нечего сравнивать
        print("Target user has no watched items present in the training data.")
        return pd.DataFrame(columns=['user_id','similarity'])
    
    # Для бинарной матрицы "dot product" с целевыми item'ами = количество общих просмотренных фильмов
    # X[:, items_idx] — (n_users x len(items_idx)) sparse, sum(axis=1) -> overlap per user
    overlap = X[:, items_idx].sum(axis=1).A1  # .A1 -> 1D numpy array

    # Нормы (euclidean) для пользователей: sqrt(sum of squares). Для бинарной матрицы — sqrt(counts)
    user_norms = np.sqrt(X.sum(axis=1).A1)
    target_norm = np.sqrt(len(items_idx))

    # безопасное деление, исключаем нули в норме
    similarities = np.zeros_like(overlap, dtype=np.float32)
    nonzero = user_norms > 0
    similarities[nonzero] = overlap[nonzero] / (user_norms[nonzero] * target_norm)

    # обнулим самого пользователя (если он присутствует в обучении)
    if target_user_id in user_to_idx:
        similarities[user_to_idx[target_user_id]] = 0.0

    
    k = min(top_n, n_users - 1)
    idx_part = np.argpartition(-similarities, k)[:k]          # k индексов с наибольшими значениями (неупорядоченных)
    idx_sorted = idx_part[np.argsort(-similarities[idx_part])]  # сортируем эти k
    top_user_ids = uniq_users[idx_sorted]
    top_sims = similarities[idx_sorted]

    return pd.DataFrame({'user_id': top_user_ids, 'similarity': top_sims})

In [107]:
recommended_similar = get_similar_users_by_overlap(
    X=X,
    uniq_users=uniq_users,
    item_to_idx=item_to_idx,
    target_user_id=random_user_id,
    watched_items=watched_items,   # если watched_items пуст — передавай None, функция попробует найти в train
    top_n=50
)

print(recommended_similar.head(50))

    user_id  similarity
0    357824    0.416667
1     65110    0.408248
2    258248    0.408248
3    784266    0.408248
4    302893    0.385758
5   1058848    0.385758
6    283293    0.385758
7    351149    0.385758
8    763400    0.369274
9   1056189    0.365148
10   754050    0.365148
11   719500    0.365148
12   625241    0.365148
13   793455    0.365148
14    21293    0.365148
15   507803    0.365148
16   594370    0.360844
17   515466    0.360844
18   534820    0.353553
19    96945    0.353553
20   448952    0.353553
21   930197    0.353553
22   181115    0.353553
23   298673    0.353553
24   481412    0.353553
25   803845    0.353553
26   643035    0.353553
27   640826    0.353553
28   173078    0.353553
29   333010    0.353553
30  1070343    0.353553
31   611211    0.353553
32   817795    0.353553
33   975701    0.353553
34   240984    0.353553
35   523162    0.353553
36   422750    0.353553
37   119890    0.353553
38   490537    0.353553
39   536510    0.353553
40   935965    0

Надо взять самого близкого и порекомендовать